In [1]:
# =====================================================================
# CÉLULA 1 — Importações
# =====================================================================
import hashlib
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

ENCODING_AFD = "iso-8859-1"   # Anexo V, Portaria MTP 671/2021, item 2

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

In [2]:
# =====================================================================
# CÉLULA 2 — Caminhos dinâmicos
# Funciona em notebook, script .py e executável gerado pelo PyInstaller
# =====================================================================
def get_base_dir() -> Path:
    """Diretório base do programa, independente de onde ele está rodando."""
    # 1) Executável do PyInstaller
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent

    # 2) Notebook no VSCode
    vsc = globals().get("__vsc_ipynb_file__")
    if vsc:
        return Path(vsc).resolve().parent

    # 3) Script .py comum
    arquivo = globals().get("__file__")
    if arquivo:
        return Path(arquivo).resolve().parent

    # 4) Jupyter/Colab
    return Path.cwd()


BASE_DIR = get_base_dir()


def resolver_pasta_arquivos(nome: str = "arquivos") -> Path:
    """
    Pasta onde ficam os AFD.
    Ordem de prioridade:
      1. Variável de ambiente AFD_PASTA
      2. <pasta do executável>/arquivos
      3. <_MEIPASS>/arquivos  (PyInstaller --onefile, se empacotado como dado)
      4. cria a pasta ao lado do executável
    """
    env = os.getenv("AFD_PASTA")
    if env:
        return Path(env).expanduser().resolve()

    candidatos = [BASE_DIR / nome]

    meipass = getattr(sys, "_MEIPASS", None)
    if meipass:
        candidatos.append(Path(meipass) / nome)

    for candidato in candidatos:
        if candidato.is_dir():
            return candidato

    destino = BASE_DIR / nome
    destino.mkdir(parents=True, exist_ok=True)
    return destino


PASTA_ARQUIVOS = resolver_pasta_arquivos()

print("BASE_DIR       :", BASE_DIR)
print("PASTA_ARQUIVOS :", PASTA_ARQUIVOS)
print("Existe         :", PASTA_ARQUIVOS.is_dir())

BASE_DIR       : \\AGROSRV\Automacoes\Marcacao_Pontos
PASTA_ARQUIVOS : \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos
Existe         : True


In [3]:
# =====================================================================
# CÉLULA 3 — Arquivos AFD encontrados
# =====================================================================
EXTENSOES_AFD = {"", ".txt", ".afd", ".dat"}


def parece_afd(caminho: Path) -> bool:
    """Confere se a 1ª linha tem o formato de cabeçalho do AFD."""
    try:
        with caminho.open("r", encoding=ENCODING_AFD, errors="replace") as f:
            primeira = f.readline().strip()
    except OSError:
        return False
    return len(primeira) >= 10 and primeira[:9].isdigit() and primeira[9] == "1"


def listar_arquivos_afd(pasta: Path | str | None = None) -> list[Path]:
    pasta = Path(pasta or PASTA_ARQUIVOS)
    if not pasta.is_dir():
        return []
    return [
        item for item in sorted(pasta.iterdir())
        if item.is_file()
        and item.suffix.lower() in EXTENSOES_AFD
        and parece_afd(item)
    ]


ARQUIVOS = listar_arquivos_afd()
print(f"{len(ARQUIVOS)} arquivo(s) AFD em {PASTA_ARQUIVOS}")
for arq in ARQUIVOS:
    print(f"  - {arq.name}  ({arq.stat().st_size / 1024:.1f} KB)")

1 arquivo(s) AFD em \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos
  - AFDBR512021002936436585411000133REP_P.txt  (113.7 KB)


In [4]:
# =====================================================================
# CÉLULA 4 — Leiaute oficial (posições 1-based, Anexo V)
# =====================================================================
CAMPOS: dict[str, dict[str, tuple[int, int]]] = {
    "1": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "tipo_id_empregador": (11, 1),
        "cnpj_cpf_empregador": (12, 14),
        "cno_caepf": (26, 14),
        "razao_social": (40, 150),
        "num_fabricacao_inpi": (190, 17),
        "data_inicial": (207, 10),
        "data_final": (217, 10),
        "dh_geracao": (227, 24),
        "versao": (251, 3),
        "tipo_id_fabricante": (254, 1),
        "cnpj_cpf_fabricante": (255, 14),
        "modelo": (269, 30),
        "crc": (299, 4),
    },
    "2": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_gravacao": (11, 24),
        "cpf_responsavel": (35, 14),
        "tipo_id_empregador": (49, 1),
        "cnpj_cpf_empregador": (50, 14),
        "cno_caepf": (64, 14),
        "razao_social": (78, 150),
        "local_prestacao": (228, 100),
        "crc": (328, 4),
    },
    "3": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_marcacao": (11, 24),
        "cpf": (35, 12),
        "crc": (47, 4),
    },
    "4": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_antes_ajuste": (11, 24),
        "dh_ajustada": (35, 24),
        "cpf_responsavel": (59, 11),
        "crc": (70, 4),
    },
    "5": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_gravacao": (11, 24),
        "operacao": (35, 1),
        "cpf": (36, 12),
        "nome": (48, 52),
        "dados_identificacao": (100, 4),
        "cpf_responsavel": (104, 11),
        "crc": (115, 4),
    },
    "6": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_gravacao": (11, 24),
        "evento": (35, 2),
    },
    "7": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_marcacao": (11, 24),
        "cpf": (35, 12),
        "dh_gravacao": (47, 24),
        "coletor": (71, 2),
        "online_offline": (73, 1),
        "hash": (74, 64),
    },
}

TAMANHO_REGISTRO = {"1": 302, "2": 331, "3": 50,
                    "4": 73, "5": 118, "7": 137, "9": 64}

COLETOR = {
    "01": "aplicativo mobile",
    "02": "browser (navegador)",
    "03": "aplicativo desktop",
    "04": "dispositivo eletrônico",
    "05": "outro dispositivo eletrônico",
}

EVENTO_REP = {
    "01": "abertura do REP por manutenção/violação",
    "02": "retorno de energia",
    "03": "introdução de dispositivo externo (Porta Fiscal)",
    "04": "retirada de dispositivo externo (Porta Fiscal)",
    "05": "emissão da Relação Instantânea de Marcações",
    "06": "erro de impressão",
    "07": "disponibilidade de serviço",
    "08": "indisponibilidade de serviço",
}

OPERACAO = {"I": "inclusão", "A": "alteração", "E": "exclusão"}

QTD_TRAILER = {"2": (10, 9), "3": (19, 9), "4": (28, 9),
               "5": (37, 9), "6": (46, 9), "7": (55, 9)}

ASSINATURA_P7S = "ASSINATURA_DIGITAL_EM_ARQUIVO_P7S"
TIPOS_COM_CRC = ("1", "2", "3", "4", "5")

In [5]:
# =====================================================================
# CÉLULA 5 — Utilidades de campo, datas e CRC-16
# =====================================================================
def extrair(linha: str, posicao: int, tamanho: int) -> str:
    """Extrai campo usando posição inicial 1-based, como no Anexo V."""
    inicio = posicao - 1
    return linha[inicio:inicio + tamanho]


def limpar(valor: str) -> str:
    return (valor or "").strip()


def somente_digitos(valor: str) -> str:
    return re.sub(r"\D", "", valor or "")


def formatar_cpf(cpf: str) -> str:
    d = somente_digitos(cpf)[-11:]
    return f"{d[:3]}.{d[3:6]}.{d[6:9]}-{d[9:]}" if len(d) == 11 else limpar(cpf)


def formatar_cnpj(cnpj: str) -> str:
    d = somente_digitos(cnpj)
    if len(d) == 14:
        return f"{d[:2]}.{d[2:5]}.{d[5:8]}/{d[8:12]}-{d[12:]}"
    return limpar(cnpj)


def parse_data_hora(valor: str) -> datetime | None:
    """Campo DH do Anexo V: 2021-04-27T16:44:00-0300"""
    texto = limpar(valor)
    if not texto:
        return None
    for fmt in ("%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(texto, fmt)
        except ValueError:
            continue
    try:
        return pd.to_datetime(texto, format="ISO8601").to_pydatetime()
    except Exception:
        return None


def parse_data(valor: str):
    texto = limpar(valor)
    if not texto:
        return None
    try:
        return datetime.strptime(texto, "%Y-%m-%d").date()
    except ValueError:
        return None


def crc16_kermit(dados: bytes, crc: int = 0x0000) -> int:
    """
    CRC-16/CCITT-TRUE (KERMIT), exigido pela Portaria 671 para AFD de REP-A e REP-P.
    Vetor de teste: b'123456789' -> 0x2189
    """
    for byte in dados:
        crc ^= byte
        for _ in range(8):
            crc = (crc >> 1) ^ 0x8408 if crc & 0x0001 else crc >> 1
    return crc & 0xFFFF


def calcular_crc(linha: str, tamanho_crc: int = 4) -> str:
    return f"{crc16_kermit(linha[:-tamanho_crc].encode(ENCODING_AFD, errors='replace')):04X}"


def crc_valido(linha: str, tamanho_crc: int = 4) -> bool:
    if len(linha) <= tamanho_crc:
        return False
    return calcular_crc(linha, tamanho_crc) == limpar(linha[-tamanho_crc:]).upper()

In [6]:
# =====================================================================
# CÉLULA 6 — Parser tipo "1" (cabeçalho)
# =====================================================================
def parse_tipo_1(linha: str) -> dict:
    c = CAMPOS["1"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    id_emp = campo("tipo_id_empregador")
    documento = somente_digitos(campo("cnpj_cpf_empregador"))

    return {
        "tipo": "1",
        "nsr": campo("nsr"),
        "tipo_id_empregador": id_emp,
        "empregador_documento": formatar_cnpj(documento) if id_emp == "1" else formatar_cpf(documento),
        "empregador_razao_social": campo("razao_social"),
        "cno_caepf": campo("cno_caepf"),
        "numero_fabricacao_inpi": campo("num_fabricacao_inpi"),
        "data_inicial": parse_data(campo("data_inicial")),
        "data_final": parse_data(campo("data_final")),
        "dh_geracao": parse_data_hora(campo("dh_geracao")),
        "versao_leiaute": campo("versao"),
        "fabricante_documento": campo("cnpj_cpf_fabricante"),
        "modelo_rep": campo("modelo"),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha) if len(linha) >= 302 else None,
    }

In [7]:
# =====================================================================
# CÉLULA 7 — Parser tipo "2" (inclusão/alteração da empresa no REP)
# =====================================================================
def parse_tipo_2(linha: str) -> dict:
    c = CAMPOS["2"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    return {
        "tipo": "2",
        "nsr": campo("nsr"),
        "dh_gravacao": parse_data_hora(campo("dh_gravacao")),
        "cpf_responsavel": formatar_cpf(campo("cpf_responsavel")),
        "tipo_id_empregador": campo("tipo_id_empregador"),
        "empregador_documento": campo("cnpj_cpf_empregador"),
        "cno_caepf": campo("cno_caepf"),
        "empregador_razao_social": campo("razao_social"),
        "local_prestacao": campo("local_prestacao"),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha),
    }

In [8]:
# =====================================================================
# CÉLULA 8 — Parser tipo "3" (marcação de ponto REP-C e REP-A)
# =====================================================================
def parse_tipo_3(linha: str) -> dict:
    c = CAMPOS["3"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))
    dh_raw = campo("dh_marcacao")

    return {
        "tipo": "3",
        "nsr": campo("nsr"),
        "dh_marcacao_raw": dh_raw,
        "dh_marcacao": parse_data_hora(dh_raw),
        "cpf": somente_digitos(campo("cpf")),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha),
    }

In [9]:
# =====================================================================
# CÉLULA 9 — Parser tipo "4" (ajuste do relógio)
# =====================================================================
def parse_tipo_4(linha: str) -> dict:
    c = CAMPOS["4"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    antes = parse_data_hora(campo("dh_antes_ajuste"))
    depois = parse_data_hora(campo("dh_ajustada"))
    delta = (depois - antes).total_seconds() if antes and depois else None

    return {
        "tipo": "4",
        "nsr": campo("nsr"),
        "dh_antes_ajuste": antes,
        "dh_ajustada": depois,
        "delta_segundos": delta,
        "cpf_responsavel": formatar_cpf(campo("cpf_responsavel")),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha),
    }

In [10]:
# =====================================================================
# CÉLULA 10 — Parser tipo "5" (inclusão/alteração/exclusão de empregado)
# =====================================================================
def parse_tipo_5(linha: str) -> dict:
    c = CAMPOS["5"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))
    operacao = campo("operacao").upper()

    return {
        "tipo": "5",
        "nsr": campo("nsr"),
        "dh_gravacao": parse_data_hora(campo("dh_gravacao")),
        "operacao": operacao,
        "operacao_desc": OPERACAO.get(operacao, operacao),
        "cpf": somente_digitos(campo("cpf")),
        "nome": campo("nome"),
        "dados_identificacao": campo("dados_identificacao"),
        "cpf_responsavel": formatar_cpf(campo("cpf_responsavel")),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha),
    }

In [11]:
# =====================================================================
# CÉLULA 11 — Parser tipo "6" (eventos sensíveis do REP)
# Observação: o Anexo V não fixa CRC-16 para o tipo 6; o resíduo é
# preservado em "complemento" para conferência manual.
# =====================================================================
def parse_tipo_6(linha: str) -> dict:
    c = CAMPOS["6"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    evento = campo("evento")
    resto = linha[36:]                      # após o campo 4 (posições 35-36)
    complemento = resto
    crc = ""
    if re.fullmatch(r"[0-9A-Fa-f]{4}", resto[-4:] or ""):
        complemento, crc = resto[:-4], resto[-4:]

    return {
        "tipo": "6",
        "nsr": campo("nsr"),
        "dh_gravacao": parse_data_hora(campo("dh_gravacao")),
        "evento": evento,
        "evento_desc": EVENTO_REP.get(evento, evento),
        "complemento": limpar(complemento),
        "crc": crc,
    }

In [12]:
# =====================================================================
# CÉLULA 12 — Parser tipo "7" (marcação de ponto REP-P)
# =====================================================================
def parse_tipo_7(linha: str) -> dict:
    c = CAMPOS["7"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    dh_marc_raw = campo("dh_marcacao")
    dh_grav_raw = campo("dh_gravacao")
    coletor = campo("coletor")
    ligado = campo("online_offline")
    dh_marc = parse_data_hora(dh_marc_raw)
    dh_grav = parse_data_hora(dh_grav_raw)

    return {
        "tipo": "7",
        "nsr": campo("nsr"),
        "dh_marcacao_raw": dh_marc_raw,
        "dh_marcacao": dh_marc,
        "cpf": somente_digitos(campo("cpf")),
        "dh_gravacao_raw": dh_grav_raw,
        "dh_gravacao": dh_grav,
        "coletor": coletor,
        "coletor_desc": COLETOR.get(coletor, coletor),
        "online_offline": ligado,
        "modo": "off-line" if ligado == "1" else "on-line",
        "atraso_gravacao_seg": (dh_grav - dh_marc).total_seconds() if dh_marc and dh_grav else None,
        "hash": campo("hash"),
        "hash_ok": None,        # preenchido na validação da cadeia
        "hash_calculado": None,
    }

In [13]:
# =====================================================================
# CÉLULA 13 — Parser tipo "9" (trailer) e assinatura digital
# =====================================================================
def parse_tipo_9(linha: str) -> dict:
    # Tipo 9: "999999999" em 1-9 e o dígito "9" na posição 64
    reg = {"tipo": "9", "marcador": limpar(extrair(linha, 1, 9))}
    for tipo, (posicao, tamanho) in QTD_TRAILER.items():
        texto = somente_digitos(extrair(linha, posicao, tamanho))
        reg[f"qtd_tipo_{tipo}"] = int(texto) if texto else 0

    fechamento = limpar(extrair(linha, 64, 1))
    reg["fechamento"] = fechamento
    reg["fechamento_ok"] = fechamento == "9"
    return reg


def parse_assinatura(linha: str) -> dict:
    texto = limpar(linha)
    return {
        "tipo": "ASSINATURA",
        "conteudo": texto,
        "p7s_padrao_ok": texto == ASSINATURA_P7S,
        "tamanho": len(linha),
    }

In [14]:
# =====================================================================
# CÉLULA 14 — Dispatcher de registro e leitura do arquivo
# =====================================================================
PARSERS = {
    "1": parse_tipo_1,
    "2": parse_tipo_2,
    "3": parse_tipo_3,
    "4": parse_tipo_4,
    "5": parse_tipo_5,
    "6": parse_tipo_6,
    "7": parse_tipo_7,
    "9": parse_tipo_9,
}


def parse_linha(linha: str, numero_linha: int | None = None) -> dict:
    linha = linha.rstrip("\r\n")

    if not linha.strip():
        return {"tipo": "BRANCO", "linha_arquivo": numero_linha}

    # Trailer: identificado pelo marcador, pois o tipo não está na posição 10
    if limpar(linha[:9]) == "999999999":
        reg = parse_tipo_9(linha)

    elif limpar(linha).startswith(ASSINATURA_P7S[:20]):
        reg = parse_assinatura(linha)

    elif len(linha) < 10:
        return {"tipo": "INVALIDO", "linha_arquivo": numero_linha,
                "erro": f"linha com apenas {len(linha)} caracteres", "bruto": linha}

    else:
        tipo = linha[9]
        parser = PARSERS.get(tipo)
        if parser is None:
            return {"tipo": tipo, "linha_arquivo": numero_linha,
                    "erro": f"tipo de registro {tipo!r} não previsto no Anexo V",
                    "bruto": linha}
        try:
            reg = parser(linha)
        except Exception as exc:
            return {"tipo": tipo, "linha_arquivo": numero_linha,
                    "erro": f"{type(exc).__name__}: {exc}", "bruto": linha}

    reg["linha_arquivo"] = numero_linha
    reg["tamanho_linha"] = len(linha)
    reg["bruto"] = linha
    return reg


def ler_afd(caminho: Path | str) -> dict:
    caminho = Path(caminho)
    with caminho.open("r", encoding=ENCODING_AFD, errors="replace", newline="") as f:
        texto = f.read()

    linhas = texto.splitlines()
    registros = [parse_linha(linha, i)
                 for i, linha in enumerate(linhas, start=1)]
    registros = [r for r in registros if r["tipo"] != "BRANCO"]

    return {
        "arquivo": caminho,
        "total_linhas": len(linhas),
        "registros": registros,
    }

In [15]:
# =====================================================================
# CÉLULA 15 — Validações do arquivo
# =====================================================================
def validar_nsr(registros: list[dict]) -> list[str]:
    problemas = []
    sequencia = []
    for reg in registros:
        if reg["tipo"] in ("ASSINATURA", "INVALIDO", "BRANCO") or reg["tipo"] == "1":
            continue
        digitos = somente_digitos(reg.get("nsr", ""))
        if digitos:
            sequencia.append((int(digitos), reg))

    for (n1, r1), (n2, r2) in zip(sequencia, sequencia[1:]):
        if n2 != n1 + 1:
            problemas.append(
                f"NSR fora de sequência: {n1} (linha {r1['linha_arquivo']}) "
                f"-> {n2} (linha {r2['linha_arquivo']})"
            )
    return problemas


def validar_trailer(registros: list[dict]) -> list[str]:
    trailer = next((r for r in registros if r["tipo"] == "9"), None)
    if not trailer:
        return ["Trailer (tipo 9) não encontrado."]

    problemas = []
    for tipo in ("2", "3", "4", "5", "6", "7"):
        declarado = trailer.get(f"qtd_tipo_{tipo}", 0)
        real = sum(1 for r in registros if r["tipo"] == tipo)
        if declarado != real:
            problemas.append(
                f"Trailer declara {declarado} registro(s) tipo {tipo}; arquivo contém {real}."
            )
    return problemas


def validar_crc_registros(registros: list[dict]) -> list[str]:
    problemas = []
    for reg in registros:
        if reg["tipo"] not in TIPOS_COM_CRC:
            continue
        bruto = reg.get("bruto", "")
        if len(bruto) < 10:
            continue
        if not crc_valido(bruto):
            problemas.append(
                f"CRC-16 inválido no NSR {reg.get('nsr')} tipo {reg['tipo']} "
                f"(linha {reg['linha_arquivo']})."
            )
    return problemas


def validar_hash_tipo_7(registros: list[dict], hash_inicial: str = "") -> list[str]:
    """
    Cadeia SHA-256 do tipo 7, conforme Anexo V:
    NSR + tipo + DH marcação + CPF + DH gravação + coletor + on/off + hash anterior.
    """
    problemas = []
    anterior = hash_inicial

    for reg in registros:
        if reg["tipo"] != "7" or "hash" not in reg:
            continue

        base = "".join([
            reg["nsr"],
            reg["tipo"],
            reg["dh_marcacao_raw"],
            reg["cpf"],
            reg["dh_gravacao_raw"],
            reg["coletor"],
            reg["online_offline"],
            anterior,
        ])
        calculado = hashlib.sha256(base.encode(
            ENCODING_AFD, errors="replace")).hexdigest()

        reg["hash_calculado"] = calculado
        reg["hash_ok"] = calculado.upper() == reg["hash"].upper()

        if not reg["hash_ok"]:
            problemas.append(
                f"Hash divergente no NSR {reg['nsr']} (linha {reg['linha_arquivo']})."
            )
        anterior = reg["hash"]

    return problemas


def validar_estrutura(registros: list[dict]) -> list[str]:
    problemas = []
    tipos = [r["tipo"] for r in registros]

    if not tipos or tipos[0] != "1":
        problemas.append(
            "O arquivo não começa com registro tipo 1 (cabeçalho).")

    if "9" not in tipos:
        problemas.append("O arquivo não contém registro tipo 9 (trailer).")
    else:
        posteriores = [t for t in tipos[tipos.index(
            "9") + 1:] if t != "ASSINATURA"]
        if posteriores:
            problemas.append(
                f"Existem registros após o trailer: {sorted(set(posteriores))}.")

    for reg in registros:
        if "erro" in reg:
            problemas.append(f"Linha {reg['linha_arquivo']}: {reg['erro']}")

    return problemas

In [16]:
# =====================================================================
# CÉLULA 16 — Consolidação em DataFrames
# =====================================================================
def filtrar_tipo(registros: list[dict], tipo: str) -> list[dict]:
    return [r for r in registros if r["tipo"] == tipo]


def dataframe_tipo(registros: list[dict], tipo: str) -> pd.DataFrame:
    linhas = filtrar_tipo(registros, tipo)
    if not linhas:
        return pd.DataFrame()
    df = pd.DataFrame(linhas)

    if "cpf" in df.columns:
        df["cpf_formatado"] = df["cpf"].apply(formatar_cpf)

    if tipo == "7" and "dh_marcacao" in df.columns:
        df = df.sort_values(["cpf", "dh_marcacao"],
                            na_position="last").reset_index(drop=True)
        df["data"] = df["dh_marcacao"].dt.date
        df["hora"] = df["dh_marcacao"].dt.strftime("%H:%M:%S")

    return df


def montar_dataframes(registros: list[dict]) -> dict[str, pd.DataFrame]:
    return {
        "cabecalho": dataframe_tipo(registros, "1"),
        "empresa_rep": dataframe_tipo(registros, "2"),
        "marcacoes_repca": dataframe_tipo(registros, "3"),
        "ajustes_relogio": dataframe_tipo(registros, "4"),
        "empregados": dataframe_tipo(registros, "5"),
        "eventos": dataframe_tipo(registros, "6"),
        "marcacoes_repp": dataframe_tipo(registros, "7"),
        "trailer": dataframe_tipo(registros, "9"),
        "assinatura": dataframe_tipo(registros, "ASSINATURA"),
    }

In [17]:
# =====================================================================
# CÉLULA 17 — Resumo do arquivo
# =====================================================================
def resumo(resultado: dict) -> dict:
    registros = resultado["registros"]
    df_cab = dataframe_tipo(registros, "1")
    marcacoes = filtrar_tipo(registros, "7") or filtrar_tipo(registros, "3")

    cab = df_cab.iloc[0].to_dict() if not df_cab.empty else {}
    cpfs = sorted({r["cpf"] for r in marcacoes if r.get("cpf")})

    return {
        "arquivo": resultado["arquivo"].name,
        "empregador": cab.get("empregador_razao_social"),
        "documento_empregador": cab.get("empregador_documento"),
        "versao_leiaute": cab.get("versao_leiaute"),
        "periodo": f"{cab.get('data_inicial')} a {cab.get('data_final')}",
        "gerado_em": cab.get("dh_geracao"),
        "total_linhas": resultado["total_linhas"],
        "total_registros": len(registros),
        "marcacoes": len(marcacoes),
        "empregados_com_marcacao": len(cpfs),
        "contagem_por_tipo": pd.Series([r["tipo"] for r in registros]).value_counts().to_dict(),
    }


def imprimir_resumo(resultado: dict) -> None:
    info = resumo(resultado)
    print("=" * 70)
    print("RESUMO DO ARQUIVO AFD")
    print("=" * 70)
    for chave, valor in info.items():
        if chave == "contagem_por_tipo":
            print(f"{chave:26}: {valor}")
        else:
            print(f"{chave:26}: {valor}")

    registros = resultado["registros"]
    problemas = (
        validar_estrutura(registros)
        + validar_nsr(registros)
        + validar_trailer(registros)
        + validar_crc_registros(registros)
        + validar_hash_tipo_7(registros)
    )

    print("-" * 70)
    if problemas:
        print(f"{len(problemas)} inconsistência(s) encontrada(s):")
        for p in problemas[:30]:
            print("  •", p)
        if len(problemas) > 30:
            print(f"  ... e mais {len(problemas) - 30}.")
    else:
        print("Nenhuma inconsistência encontrada.")
    print("=" * 70)

In [18]:
# =====================================================================
# CÉLULA 18 — Exportação dos resultados
# =====================================================================
def normalizar_para_excel(df: pd.DataFrame) -> pd.DataFrame:
    """Remove timezone — o Excel não aceita datetime com offset."""
    df = df.copy()
    for col in df.columns:
        serie = df[col]
        if isinstance(serie.dtype, pd.DatetimeTZDtype):
            df[col] = serie.dt.tz_localize(None)
        elif serie.dtype == object:
            df[col] = serie.map(
                lambda v: v.replace(tzinfo=None)
                if isinstance(v, datetime) and v.tzinfo else v
            )
    return df


def exportar_excel(resultado: dict, destino: Path | str | None = None) -> Path:
    destino = Path(destino or (
        resultado["arquivo"].parent /
        f"{resultado['arquivo'].stem}_analisado.xlsx"
    ))
    dfs = montar_dataframes(resultado["registros"])

    resumo_df = pd.DataFrame([resumo(resultado)]).T.reset_index()
    resumo_df.columns = ["campo", "valor"]
    resumo_df["valor"] = resumo_df["valor"].map(
        lambda v: str(v) if isinstance(v, (dict, list)) else v
    )
    resumo_df = normalizar_para_excel(resumo_df)

    with pd.ExcelWriter(destino, engine="openpyxl",
                        datetime_format="dd/mm/yyyy hh:mm:ss") as writer:
        resumo_df.to_excel(writer, sheet_name="resumo", index=False)

        for nome, df in dfs.items():
            if not df.empty:
                normalizar_para_excel(df).to_excel(
                    writer, sheet_name=nome[:31], index=False
                )

    return destino

In [19]:
# =====================================================================
# CÉLULA 19 — Pipeline: lê todos os arquivos da pasta e exporta
# =====================================================================
def processar_arquivo(caminho: Path, exportar: bool = True) -> dict:
    resultado = ler_afd(caminho)
    imprimir_resumo(resultado)

    dfs = montar_dataframes(resultado["registros"])
    resultado["dataframes"] = dfs

    if not dfs["marcacoes_repp"].empty:
        print("\nAmostra das marcações REP-P:")
        print(dfs["marcacoes_repp"][["nsr", "cpf_formatado",
              "dh_marcacao", "modo", "coletor_desc"]].head())
    elif not dfs["marcacoes_repca"].empty:
        print("\nAmostra das marcações REP-C/REP-A:")
        print(dfs["marcacoes_repca"]
              [["nsr", "cpf_formatado", "dh_marcacao"]].head())

    if exportar:
        print("\nPlanilha gerada:", exportar_excel(resultado).name)

    return resultado


RESULTADOS = []
for arquivo in ARQUIVOS:
    print()
    RESULTADOS.append(processar_arquivo(arquivo))

# Acesso rápido ao último arquivo processado
if RESULTADOS:
    ultimo = RESULTADOS[-1]
    df_cabecalho = ultimo["dataframes"]["cabecalho"]
    df_marcacoes = ultimo["dataframes"]["marcacoes_repp"]


RESUMO DO ARQUIVO AFD
arquivo                   : AFDBR512021002936436585411000133REP_P.txt
empregador                : TOKUNAGA COMERCIO DE RACOES LTDA
documento_empregador      : 36.585.411/0001-33
versao_leiaute            : 003
periodo                   : 2026-09-26 a 2026-10-25
gerado_em                 : 2026-10-26 12:40:00-03:00
total_linhas              : 837
total_registros           : 837
marcacoes                 : 834
empregados_com_marcacao   : 11
contagem_por_tipo         : {'7': 834, '1': 1, '9': 1, 'ASSINATURA': 1}
----------------------------------------------------------------------
1 inconsistência(s) encontrada(s):
  • Hash divergente no NSR 000049866 (linha 2).

Amostra das marcações REP-P:
         nsr   cpf_formatado               dh_marcacao     modo       coletor_desc
0  000049870  092.819.288-17 2026-09-26 07:16:00-03:00  on-line  aplicativo mobile
1  000049880  092.819.288-17 2026-09-26 11:44:00-03:00  on-line  aplicativo mobile
2  000049887  092.819.288-17 

In [20]:
# =====================================================================
# CÉLULA 20 — Diagnóstico da fórmula do hash do tipo 7
# =====================================================================
def diagnosticar_hash(registros: list[dict], limite: int = 6) -> pd.DataFrame:
    marcs = [r for r in registros if r["tipo"] == "7"]
    if not marcs:
        return pd.DataFrame()

    amostra = marcs[:limite]
    testes = []

    for nome_prev, prev_inicial in (("vazio", ""), ("64 zeros", "0" * 64)):
        for modo_base in ("campos + anterior", "linha até 73"):
            for modo_cpf in ("campo_12", "11 dígitos"):
                for modo_ant in ("como no campo", "minúsculo"):
                    anterior, falhou = prev_inicial, None

                    for reg in amostra:
                        bruto = reg["bruto"]
                        cpf = extrair(bruto, 35, 12)
                        if modo_cpf == "11 dígitos":
                            cpf = somente_digitos(cpf)[-11:]

                        ant = anterior.lower() if modo_ant == "minúsculo" else anterior

                        if modo_base == "linha até 73":
                            base = bruto[:73] + ant
                        else:
                            base = "".join([
                                extrair(bruto, 1, 9), "7", extrair(
                                    bruto, 11, 24),
                                cpf, extrair(bruto, 47, 24),
                                extrair(bruto, 71, 2), extrair(
                                    bruto, 73, 1), ant,
                            ])

                        calc = hashlib.sha256(
                            base.encode(ENCODING_AFD, errors="replace")
                        ).hexdigest()

                        if calc.lower() != reg["hash"].lower():
                            falhou = reg["nsr"]
                            break
                        anterior = reg["hash"]

                    testes.append({
                        "hash_anterior_inicial": nome_prev,
                        "base": modo_base,
                        "cpf": modo_cpf,
                        "hash_anterior": modo_ant,
                        "cadeia_ok": falhou is None,
                        "falhou_no_nsr": falhou or "—",
                    })

    return pd.DataFrame(testes)


DIAGNOSTICO_HASH = diagnosticar_hash(
    RESULTADOS[-1]["registros"]) if RESULTADOS else pd.DataFrame()
DIAGNOSTICO_HASH

,hash_anterior_inicial,base,cpf,hash_anterior,cadeia_ok,falhou_no_nsr
0,vazio,campos + anterior,campo_12,como no campo,False,000049866
1,vazio,campos + anterior,campo_12,minúsculo,False,000049866
2,vazio,campos + anterior,11 dígitos,como no campo,False,000049866
3,vazio,campos + anterior,11 dígitos,minúsculo,False,000049866
4,vazio,linha até 73,campo_12,como no campo,False,000049866
5,vazio,linha até 73,campo_12,minúsculo,False,000049866
6,vazio,linha até 73,11 dígitos,como no campo,False,000049866
7,vazio,linha até 73,11 dígitos,minúsculo,False,000049866
8,64 zeros,campos + anterior,campo_12,como no campo,False,000049866
9,64 zeros,campos + anterior,campo_12,minúsculo,False,000049866
